# 07 - LLM-Generated Analyst Narratives

Member 3 (Explainability & Narrative Engineer)
---

## Purpose

SHAP tells us *which features* drove a prediction and *by how much*, but it does so in numbers.  A bank analyst, an auditor, or a portfolio manager cannot easily read a SHAP vector.  This notebook uses an LLM (**OpenAI GPT-4o-mini**) to translate SHAP values into a plain-English, analyst-style report - one for every company in a curated sample of test-set predictions.

## The API Key

This notebook expects a plain text file at the project root called `openai_api_key.txt` containing your OpenAI key (nothing else).  The file is already gitignored, so it is never pushed to GitHub.

## Step 1 - Setup

In [1]:
import os, sys, json, time
from pathlib import Path

sys.path.insert(0, os.path.abspath('../src'))
import config as C

import numpy as np
import pandas as pd
from openai import OpenAI, RateLimitError, APIError, APIConnectionError

# Where things go
SHAP_DIR = C.RESULTS_DIR / 'shap'
NARR_DIR = C.RESULTS_DIR / 'narratives'
NARR_DIR.mkdir(parents=True, exist_ok=True)

print('OpenAI Python SDK loaded.')
print('SHAP dir:      ', SHAP_DIR)
print('Narratives out:', NARR_DIR)

ModuleNotFoundError: No module named 'openai'

## Step 2 - Load API Key

In [ ]:
def load_openai_key():
    candidates = [C.ROOT / 'openai_api_key.txt', C.ROOT.parent / 'openai_api_key.txt']
    for path in candidates:
        if path.exists():
            print(f'Key loaded from: {path}')
            return path.read_text().strip()
    raise FileNotFoundError(
        'openai_api_key.txt not found. Create it in the project root, '
        'paste your OpenAI API key inside, save, and re-run this cell.'
    )

api_key = load_openai_key()
client = OpenAI(api_key=api_key)
print('OpenAI client ready.')

## Step 3 - Load SHAP Outputs

In [ ]:
shap_summary_path = SHAP_DIR / 'shap_top_features_xgboost.csv'
if not shap_summary_path.exists():
    raise FileNotFoundError(
        f'{shap_summary_path} missing. Run notebook 06 first - it produces this file.'
    )

shap_df = pd.read_csv(shap_summary_path)
print(f'Loaded {len(shap_df):,} SHAP rows.')
print(f'Columns: {list(shap_df.columns)}')
shap_df.head(3)

## Step 4 - Curate a Balanced Sample of 100 Rows

A random sample would be dominated by Distressed (majority class) and correct predictions (majority outcome).  We deliberately balance the sample across:

| Bucket | Target count | Why |
|---|---|---|
| Correct - Healthy | 25 | Model succeeds on a minority class |
| Correct - At-Risk | 25 | Model handles the grey zone |
| Correct - Distressed | 25 | Model catches the majority class |
| Wrong (any type) | 25 | The interesting failure modes |

This ensures the narratives (and the human study in notebook 08) cover the full behavioural range.

In [ ]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

def sample_from(df, n, replace=False):
    n = min(n, len(df))
    return df.sample(n=n, replace=replace, random_state=RANDOM_SEED)

buckets = {
    'correct_healthy':    shap_df[(shap_df.correct) & (shap_df.predicted_label == 'Healthy')],
    'correct_at_risk':    shap_df[(shap_df.correct) & (shap_df.predicted_label == 'At-Risk')],
    'correct_distressed': shap_df[(shap_df.correct) & (shap_df.predicted_label == 'Distressed')],
    'wrong':              shap_df[~shap_df.correct],
}

target_per_bucket = 25
picked = [sample_from(b, target_per_bucket) for b in buckets.values()]
sample_df = pd.concat(picked).reset_index(drop=True)

# If any bucket was short, top up randomly from the remaining rows
if len(sample_df) < 100:
    used_keys = set(zip(sample_df.company_id.astype(str), sample_df.year.astype(str)))
    remaining = shap_df[~shap_df.apply(
        lambda r: (str(r.company_id), str(r.year)) in used_keys, axis=1)]
    top_up = sample_from(remaining, 100 - len(sample_df))
    sample_df = pd.concat([sample_df, top_up]).reset_index(drop=True)

print(f'Sample size: {len(sample_df)}')
print('Composition:')
print(sample_df.groupby(['predicted_label', 'correct']).size().unstack(fill_value=0))

## Step 5 - Prompt Template

In [ ]:
SYSTEM_MESSAGE = (
    'You are a senior financial analyst. You write short, evidence-based '
    'analyst reports that translate a machine-learning model prediction and '
    'its SHAP feature attributions into plain business English for a bank '
    'credit committee.\n\n'
    'Rules:\n'
    '1. Use only the numbers I give you. Do not invent or estimate any figure.\n'
    '2. Reference each top SHAP feature by name at least once.\n'
    '3. State whether the model was confident or uncertain.\n'
    '4. If the top SHAP contributions are negative, explain what worsens the outlook. '
    'If positive, explain what supports it.\n'
    '5. Two short paragraphs, roughly 120-180 words total.\n'
    '6. Do not use bullet points, headers, or Markdown.\n'
)

def build_prompt(row):
    """Turn one shap_summary row into a user message string."""
    probs = {
        'Healthy':    row['proba_healthy'],
        'At-Risk':    row['proba_at_risk'],
        'Distressed': row['proba_distressed'],
    }
    prob_line = ', '.join(f'{k}={v:.2f}' for k, v in probs.items())

    top_features = []
    for k in (1, 2, 3):
        top_features.append(
            f'- {row[f"top{k}_feature"]}: value={row[f"top{k}_value"]:.3f}, '
            f'SHAP contribution={row[f"top{k}_shap"]:+.3f}'
        )
    features_block = '\n'.join(top_features)

    return (
        f'Company: {row["company_id"]}\n'
        f'Fiscal year: {int(row["year"]) if pd.notna(row["year"]) else "n/a"}\n'
        f'Model predicted class: {row["predicted_label"]}\n'
        f'Class probabilities: {prob_line}\n\n'
        f'Top three SHAP-attributed features (positive contribution pushes '
        f'toward the predicted class, negative pushes away):\n'
        f'{features_block}\n\n'
        f'Write the analyst report now.'
    )

# Preview one prompt
print('--- Example prompt ---')
print(build_prompt(sample_df.iloc[0]))

## Step 6 - Single API Call Helper

Wraps `client.chat.completions.create` with retry-on-rate-limit and token accounting.  Everything downstream just calls `generate_one(prompt)` and forgets about the plumbing.

In [ ]:
MODEL = 'gpt-4o-mini'
TEMPERATURE = 0.4     # low - we want consistent, evidence-based text
MAX_RETRIES = 3

# gpt-4o-mini pricing (2026, USD per 1M tokens)
PRICE_IN  = 0.15 / 1_000_000
PRICE_OUT = 0.60 / 1_000_000

def generate_one(prompt: str):
    """Return dict with narrative, usage, cost. Retries on transient errors."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                temperature=TEMPERATURE,
                messages=[
                    {'role': 'system', 'content': SYSTEM_MESSAGE},
                    {'role': 'user',   'content': prompt},
                ],
            )
            usage = resp.usage
            cost = usage.prompt_tokens * PRICE_IN + usage.completion_tokens * PRICE_OUT
            return {
                'narrative':      resp.choices[0].message.content.strip(),
                'input_tokens':   usage.prompt_tokens,
                'output_tokens':  usage.completion_tokens,
                'cost_usd':       cost,
                'error':          None,
            }
        except RateLimitError as e:
            wait = 5 * attempt
            print(f'  rate limited (attempt {attempt}) - sleeping {wait}s')
            time.sleep(wait)
        except (APIError, APIConnectionError) as e:
            wait = 3 * attempt
            print(f'  API error {type(e).__name__} (attempt {attempt}) - sleeping {wait}s')
            time.sleep(wait)
    return {'narrative': None, 'input_tokens': 0, 'output_tokens': 0,
            'cost_usd': 0.0, 'error': 'exceeded retries'}

## Step 7 - Smoke Test on 5 Rows

Run five narratives first (~$0.002).  Read them.  If quality is bad, fix the prompt before spending on all 100.

In [ ]:
smoke_sample = sample_df.head(5)

smoke_results = []
for _, row in smoke_sample.iterrows():
    prompt = build_prompt(row)
    result = generate_one(prompt)
    smoke_results.append({**row.to_dict(), 'prompt': prompt, **result})
    print(f'{row["company_id"]} {int(row["year"])} pred={row["predicted_label"]:10s} '
          f'true={row["true_label"]:10s}  '
          f'{result["input_tokens"]:>4}+{result["output_tokens"]:>4} tokens  '
          f'${result["cost_usd"]:.5f}')

print(f'\nTotal smoke-test cost: ${sum(r["cost_usd"] for r in smoke_results):.4f}')

In [ ]:
# Read one narrative to see what quality looks like
sample_out = smoke_results[0]
print(f'Company: {sample_out["company_id"]} ({sample_out["year"]})')
print(f'True: {sample_out["true_label"]}  |  Predicted: {sample_out["predicted_label"]}')
print('-' * 70)
print(sample_out['narrative'])

## Step 8 - Generate All 100 Narratives

In [ ]:
out_path = NARR_DIR / 'narratives.jsonl'

# Idempotent restart: skip rows already written
already_done = set()
if out_path.exists():
    with open(out_path) as f:
        for line in f:
            rec = json.loads(line)
            already_done.add((str(rec['company_id']), str(rec.get('year', ''))))
    print(f'Resuming: {len(already_done)} rows already written.')

total_cost = 0.0
total_in = total_out = 0
n_written = 0

with open(out_path, 'a') as f:
    for i, (_, row) in enumerate(sample_df.iterrows(), 1):
        key = (str(row['company_id']), str(row.get('year', '')))
        if key in already_done:
            continue

        prompt = build_prompt(row)
        result = generate_one(prompt)

        record = {
            'company_id':      row['company_id'],
            'year':            int(row['year']) if pd.notna(row['year']) else None,
            'true_label':      row['true_label'],
            'predicted_label': row['predicted_label'],
            'correct':         bool(row['correct']),
            'top1_feature':    row['top1_feature'],
            'top2_feature':    row['top2_feature'],
            'top3_feature':    row['top3_feature'],
            'prompt':          prompt,
            'narrative':       result['narrative'],
            'model':           MODEL,
            'input_tokens':    result['input_tokens'],
            'output_tokens':   result['output_tokens'],
            'cost_usd':        round(result['cost_usd'], 6),
            'error':           result['error'],
        }
        f.write(json.dumps(record) + '\n')
        f.flush()

        total_cost += result['cost_usd']
        total_in   += result['input_tokens']
        total_out  += result['output_tokens']
        n_written  += 1

        if i % 10 == 0 or i == len(sample_df):
            print(f'  {i:3d}/{len(sample_df)}   '
                  f'tokens in={total_in:,}  out={total_out:,}  '
                  f'cost=${total_cost:.4f}')

print(f'\nWrote {n_written} new narratives to {out_path}')
print(f'Total cost this run: ${total_cost:.4f}')

## Step 9 - Load Back and Verify

Reload from disk to confirm the JSONL is well-formed and every record has a narrative.

In [ ]:
records = [json.loads(l) for l in open(out_path)]
narr_df = pd.DataFrame(records)

print(f'Total records: {len(narr_df)}')
print(f'Missing narratives: {narr_df.narrative.isna().sum()}')
print(f'Errors: {narr_df.error.notna().sum()}')
print()
print('Composition:')
print(narr_df.groupby(["predicted_label", "correct"]).size().unstack(fill_value=0))
print()
print(f'Total spend:  ${narr_df.cost_usd.sum():.4f}')
print(f'Avg tokens:   {narr_df.input_tokens.mean():.0f} in, {narr_df.output_tokens.mean():.0f} out')

## Step 10 - Preview a Few Narratives

Read a random correct case and a random wrong case to see what the LLM produced.

In [ ]:
def show_narrative(row):
    print(f'Company:   {row.company_id} ({row.year})')
    print(f'True:      {row.true_label}   |   Predicted: {row.predicted_label}   |   '
          f'Correct: {row.correct}')
    print(f'Top SHAP:  {row.top1_feature}, {row.top2_feature}, {row.top3_feature}')
    print('-' * 70)
    print(row.narrative)
    print()

print('===== CORRECT PREDICTION =====')
show_narrative(narr_df[narr_df.correct].sample(1, random_state=1).iloc[0])

print('===== WRONG PREDICTION =====')
show_narrative(narr_df[~narr_df.correct].sample(1, random_state=1).iloc[0])

## Step 11 - Also Export a Human-Readable CSV

Handy for Notebook 08 (evaluation) and for anyone who wants to eyeball narratives in Excel.

In [ ]:
csv_out = NARR_DIR / 'narratives.csv'
narr_df.to_csv(csv_out, index=False)
print(f'Saved: {csv_out}')
print(f'Also on file: {out_path} (JSONL - preferred for programmatic use)')

## Deliverable Summary

| File | Purpose |
|---|---|
| `results/narratives/narratives.jsonl` | Every prompt + narrative + metadata + cost |
| `results/narratives/narratives.csv` | Same content in tabular form |